In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

# ==========================================
# 1. 파라미터 설정 (여기서 값을 변경하며 테스트하세요)
# ==========================================
n_pca = 2    # PCA 성분 개수
n_clusters = 4  # 군집화 개수
# ==========================================

# 2. 데이터 로드 및 초기 필터링
df_origin = pd.read_csv("zigbang_outlier.csv")
df = df_origin[df_origin['is_outlier'] == False].copy()

# 3. Kernel PCA (입지 데이터 압축)
infra_cols = [
    '세탁소_거리(m)', '카페_거리(m)', 
    '약국_거리(m)', '대형마트_거리(m)', '편의점_거리(m)',
]

scaler_infra = StandardScaler()
infra_scaled = scaler_infra.fit_transform(df[infra_cols])

kpca = KernelPCA(n_components=n_pca, kernel='rbf', gamma=None) 
pca_results = kpca.fit_transform(infra_scaled)

# PC 컬럼 생성 및 데이터프레임 추가
pc_cols = [f'PC{i+1}' for i in range(n_pca)]
for i, col in enumerate(pc_cols):
    df[col] = pca_results[:, i]

# 4. 군집화를 위한 데이터 준비 및 스케일링
# 기본 변수 리스트
base_features = ['보증금', '전용면적', '노후도', '해당층', '전체층', '엘리베이터', '남향', '풀옵션', '지하철역_거리(m)', '버스정류장_거리(m)']
# 불리언 타입을 미리 숫자로 변환
for col in ['엘리베이터', '남향', '풀옵션']:
    df[col] = df[col].astype(int)

# PCA 성분을 포함한 전체 학습 변수
cluster_input_features = base_features + pc_cols
X_for_cluster = df[cluster_input_features]

scaler_final = StandardScaler()
X_scaled = scaler_final.fit_transform(X_for_cluster)

# 5. K-Means 군집화 및 시각화
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# 시각화 (PC1, PC2 기준)
plt.figure(figsize=(10, 6))
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows 기준 (Mac은 AppleGothic)
plt.rcParams['axes.unicode_minus'] = False

# 군집별 특징 요약 출력
print(f"\n📊 [군집별 주요 변수 평균값 (K={n_clusters})]")
print(df.groupby('cluster')[['월세', '보증금', '전용면적', '노후도']].mean().round(2))

# 6. 머신러닝 학습을 위한 전처리 (원핫 인코딩)
df_encoded = pd.get_dummies(df, columns=['cluster'], prefix='cluster', dtype=int)
# 새로 생성된 cluster_n 컬럼들 찾기
cluster_dummy_cols = [col for col in df_encoded.columns if col.startswith('cluster_')]

# 최종 학습 변수 설정
final_features = base_features + pc_cols + cluster_dummy_cols
X = df_encoded[final_features]
y = df_encoded['월세']

# 7. 앙상블 모델 학습 및 평가
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
hgb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)

voting_model = VotingRegressor(estimators=[('rf', rf), ('gb', gb), ('hgb', hgb)])
voting_model.fit(X_train, y_train)

# 결과 확인
y_pred = voting_model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)

print("-" * 50)
print(f"🚀 테스트 결과 (PCA: {n_pca}개, 군집: {n_clusters}개)")
print(f"📈 R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"📉 MAE: {mean_absolute_error(y_test, y_pred):.2f}만원")
print(f"📏 RMSE: {rmse:.2f}만원")
print("-" * 50)


📊 [군집별 주요 변수 평균값 (K=4)]
            월세     보증금   전용면적   노후도
cluster                            
0        35.16  205.00  30.23  6.40
1        31.97  139.91  26.94  8.09
2        30.37  167.19  34.21  7.34
3        33.16  166.76  27.63  6.28
--------------------------------------------------
🚀 테스트 결과 (PCA: 2개, 군집: 4개)
📈 R2 Score: 0.7397
📉 MAE: 2.16만원
📏 RMSE: 2.73만원
--------------------------------------------------


<Figure size 1000x600 with 0 Axes>

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import warnings

# 경고 메시지 무시 (반복문 실행 시 깔끔한 출력을 위함)
warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 초기 필터링
df_origin = pd.read_csv("zigbang_outlier.csv")
df_base = df_origin[df_origin['is_outlier'] == False].copy()

# 인프라 컬럼 및 기본 변수 설정
infra_cols = ['세탁소_거리(m)', '카페_거리(m)', '약국_거리(m)', '대형마트_거리(m)', '편의점_거리(m)']
base_features = ['보증금', '전용면적', '노후도', '해당층', '전체층', '엘리베이터', '남향', '풀옵션', '지하철역_거리(m)', '버스정류장_거리(m)']

# 불리언 타입 변환
for col in ['엘리베이터', '남향', '풀옵션']:
    df_base[col] = df_base[col].astype(int)

# 결과를 저장할 리스트
results_list = []

print("⏳ 입지 거리 PCA + Clustering 조합 테스트를 시작합니다. 잠시만 기다려 주세요...\n")

# 2. 반복문 실행 (PCA 1~2, Cluster 2~7)
for n_pca in [1, 2, 3]:
    for n_clusters in range(2, 6):
        df_temp = df_base.copy()
        
        # --- (1) Kernel PCA ---
        scaler_infra = StandardScaler()
        infra_scaled = scaler_infra.fit_transform(df_temp[infra_cols])
        
        kpca = KernelPCA(n_components=n_pca, kernel='rbf', gamma=None) 
        pca_results = kpca.fit_transform(infra_scaled)
        
        pc_cols = [f'PC{i+1}' for i in range(n_pca)]
        for i, col in enumerate(pc_cols):
            df_temp[col] = pca_results[:, i]
            
        # --- (2) 군집화 ---
        cluster_input_features = base_features + pc_cols
        scaler_final = StandardScaler()
        X_scaled = scaler_final.fit_transform(df_temp[cluster_input_features])
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_temp['cluster'] = kmeans.fit_predict(X_scaled)
        
        # --- (3) 전처리 (원핫 인코딩) ---
        df_encoded = pd.get_dummies(df_temp, columns=['cluster'], prefix='cluster', dtype=int)
        cluster_dummy_cols = [c for c in df_encoded.columns if c.startswith('cluster_')]
        
        final_features = base_features + pc_cols + cluster_dummy_cols
        X = df_encoded[final_features]
        y = df_encoded['월세']
        
        # --- (4) 모델 학습 ---
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
        gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
        hgb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)
        
        voting_model = VotingRegressor(estimators=[('rf', rf), ('gb', gb), ('hgb', hgb)])
        voting_model.fit(X_train, y_train)
        
        # --- (5) 평가 ---
        y_pred = voting_model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)
        
        # 결과 저장
        results_list.append({
            'PCA': n_pca,
            'Cluster': n_clusters,
            'R2 Score': round(r2, 4),
            'MAE': round(mae, 2),
            'RMSE': round(rmse, 2)
        })
        print(f"✅ PCA:{n_pca}, Cluster:{n_clusters} 완료 (R2: {r2:.4f})")

# 3. 요약 결과 출력
summary_df = pd.DataFrame(results_list)
summary_df = summary_df.sort_values(by='R2 Score', ascending=False) # R2 점수 높은 순 정렬

print("\n🏆 [전체 테스트 결과 요약 (R2 높은 순)]")
print(summary_df.to_string(index=False))

# 4. 최적의 조합 안내
best = summary_df.iloc[0]
print(f"\n✨ 최적의 조합: PCA {int(best['PCA'])}개, 군집 {int(best['Cluster'])}개")
print(f"   (R2 Score: {best['R2 Score']}, MAE: {best['MAE']}만원)")

⏳ 입지 거리 PCA + Clustering 조합 테스트를 시작합니다. 잠시만 기다려 주세요...

✅ PCA:1, Cluster:2 완료 (R2: 0.7390)
✅ PCA:1, Cluster:3 완료 (R2: 0.7390)
✅ PCA:1, Cluster:4 완료 (R2: 0.7386)
✅ PCA:1, Cluster:5 완료 (R2: 0.7415)
✅ PCA:2, Cluster:2 완료 (R2: 0.7375)
✅ PCA:2, Cluster:3 완료 (R2: 0.7430)
✅ PCA:2, Cluster:4 완료 (R2: 0.7402)
✅ PCA:2, Cluster:5 완료 (R2: 0.7425)
✅ PCA:3, Cluster:2 완료 (R2: 0.7404)
✅ PCA:3, Cluster:3 완료 (R2: 0.7508)
✅ PCA:3, Cluster:4 완료 (R2: 0.7398)
✅ PCA:3, Cluster:5 완료 (R2: 0.7396)

🏆 [전체 테스트 결과 요약 (R2 높은 순)]
 PCA  Cluster  R2 Score  MAE  RMSE
   3        3    0.7508 2.13  2.67
   2        3    0.7430 2.15  2.71
   2        5    0.7425 2.14  2.72
   1        5    0.7415 2.16  2.72
   3        2    0.7404 2.15  2.73
   2        4    0.7402 2.15  2.73
   3        4    0.7398 2.17  2.73
   3        5    0.7396 2.18  2.73
   1        2    0.7390 2.18  2.74
   1        3    0.7390 2.18  2.74
   1        4    0.7386 2.18  2.74
   2        2    0.7375 2.16  2.74

✨ 최적의 조합: PCA 3개, 군집 3개
   (R2 Score: 0.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import warnings

# 경고 메시지 무시 (반복문 실행 시 깔끔한 출력을 위함)
warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 초기 필터링
df_origin = pd.read_csv("zigbang_outlier.csv")
df_base = df_origin[df_origin['is_outlier'] == False].copy()

# 인프라 컬럼 및 기본 변수 설정
infra_cols = ['세탁소_거리(m)', '카페_거리(m)', '약국_거리(m)', '대형마트_거리(m)', '편의점_거리(m)', '지하철역_거리(m)', '버스정류장_거리(m)']
base_features = ['보증금', '전용면적', '노후도', '해당층', '전체층', '엘리베이터', '남향', '풀옵션']

# 불리언 타입 변환
for col in ['엘리베이터', '남향', '풀옵션']:
    df_base[col] = df_base[col].astype(int)

# 결과를 저장할 리스트
results_list = []

print("거리변수 통합 PCA + Clustering 조합 테스트를 시작합니다. 잠시만 기다려 주세요...\n")

# 2. 반복문 실행 (PCA 1~2, Cluster 2~7)
for n_pca in [2, 3, 4, 5, 6]:
    for n_clusters in range(2, 6):
        df_temp = df_base.copy()
        
        # --- (1) Kernel PCA ---
        scaler_infra = StandardScaler()
        infra_scaled = scaler_infra.fit_transform(df_temp[infra_cols])
        
        kpca = KernelPCA(n_components=n_pca, kernel='rbf', gamma=None) 
        pca_results = kpca.fit_transform(infra_scaled)
        
        pc_cols = [f'PC{i+1}' for i in range(n_pca)]
        for i, col in enumerate(pc_cols):
            df_temp[col] = pca_results[:, i]
            
        # --- (2) 군집화 ---
        cluster_input_features = base_features + pc_cols
        scaler_final = StandardScaler()
        X_scaled = scaler_final.fit_transform(df_temp[cluster_input_features])
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_temp['cluster'] = kmeans.fit_predict(X_scaled)
        
        # --- (3) 전처리 (원핫 인코딩) ---
        df_encoded = pd.get_dummies(df_temp, columns=['cluster'], prefix='cluster', dtype=int)
        cluster_dummy_cols = [c for c in df_encoded.columns if c.startswith('cluster_')]
        
        final_features = base_features + pc_cols + cluster_dummy_cols
        X = df_encoded[final_features]
        y = df_encoded['월세']
        
        # --- (4) 모델 학습 ---
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
        gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
        hgb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)
        
        voting_model = VotingRegressor(estimators=[('rf', rf), ('gb', gb), ('hgb', hgb)])
        voting_model.fit(X_train, y_train)
        
        # --- (5) 평가 ---
        y_pred = voting_model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)
        
        # 결과 저장
        results_list.append({
            'PCA': n_pca,
            'Cluster': n_clusters,
            'R2 Score': round(r2, 4),
            'MAE': round(mae, 2),
            'RMSE': round(rmse, 2)
        })
        print(f"✅ PCA:{n_pca}, Cluster:{n_clusters} 완료 (R2: {r2:.4f})")

# 3. 요약 결과 출력
summary_df = pd.DataFrame(results_list)
summary_df = summary_df.sort_values(by='R2 Score', ascending=False) # R2 점수 높은 순 정렬

print("\n🏆 [전체 테스트 결과 요약 (R2 높은 순)]")
print(summary_df.to_string(index=False))

# 4. 최적의 조합 안내
best = summary_df.iloc[0]
print(f"\n✨ 최적의 조합: PCA {int(best['PCA'])}개, 군집 {int(best['Cluster'])}개")
print(f"   (R2 Score: {best['R2 Score']}, MAE: {best['MAE']}만원)")

거리변수 통합 PCA + Clustering 조합 테스트를 시작합니다. 잠시만 기다려 주세요...

✅ PCA:2, Cluster:2 완료 (R2: 0.7029)
✅ PCA:2, Cluster:3 완료 (R2: 0.6959)
✅ PCA:2, Cluster:4 완료 (R2: 0.6979)
✅ PCA:2, Cluster:5 완료 (R2: 0.6958)
✅ PCA:3, Cluster:2 완료 (R2: 0.7176)
✅ PCA:3, Cluster:3 완료 (R2: 0.7195)
✅ PCA:3, Cluster:4 완료 (R2: 0.7216)
✅ PCA:3, Cluster:5 완료 (R2: 0.7192)
✅ PCA:4, Cluster:2 완료 (R2: 0.7221)
✅ PCA:4, Cluster:3 완료 (R2: 0.7260)
✅ PCA:4, Cluster:4 완료 (R2: 0.7263)
✅ PCA:4, Cluster:5 완료 (R2: 0.7256)
✅ PCA:5, Cluster:2 완료 (R2: 0.7193)
✅ PCA:5, Cluster:3 완료 (R2: 0.7250)
✅ PCA:5, Cluster:4 완료 (R2: 0.7196)
✅ PCA:5, Cluster:5 완료 (R2: 0.7131)
✅ PCA:6, Cluster:2 완료 (R2: 0.7040)
✅ PCA:6, Cluster:3 완료 (R2: 0.7103)
✅ PCA:6, Cluster:4 완료 (R2: 0.7066)
✅ PCA:6, Cluster:5 완료 (R2: 0.7024)

🏆 [전체 테스트 결과 요약 (R2 높은 순)]
 PCA  Cluster  R2 Score  MAE  RMSE
   4        4    0.7263 2.20  2.80
   4        3    0.7260 2.20  2.80
   4        5    0.7256 2.21  2.81
   5        3    0.7250 2.21  2.81
   4        2    0.7221 2.20  2.82
   3  

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import warnings

# 경고 메시지 무시 (반복문 실행 시 깔끔한 출력을 위함)
warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 초기 필터링
df_origin = pd.read_csv("zigbang_outlier.csv")
df_base = df_origin[df_origin['is_outlier'] == False].copy()

# 인프라 컬럼 및 기본 변수 설정
infra_cols = ['세탁소_거리(m)', '카페_거리(m)', '약국_거리(m)', '대형마트_거리(m)', '편의점_거리(m)', '지하철역_거리(m)', '버스정류장_거리(m)', '보증금', '전용면적', '노후도', '해당층', '엘리베이터', '남향', '풀옵션']
base_features 

# 불리언 타입 변환
for col in ['엘리베이터', '남향', '풀옵션']:
    df_base[col] = df_base[col].astype(int)

# 결과를 저장할 리스트
results_list = []

print("모두 PCA + Clustering 14개의 조합 테스트를 시작합니다. 잠시만 기다려 주세요...\n")

# 2. 반복문 실행 (PCA 1~2, Cluster 2~7)
for n_pca in [2, 3, 4, 5, 6, 7, 8]:
    for n_clusters in range(2, 5):
        df_temp = df_base.copy()
        
        # --- (1) Kernel PCA ---
        scaler_infra = StandardScaler()
        infra_scaled = scaler_infra.fit_transform(df_temp[infra_cols])
        
        kpca = KernelPCA(n_components=n_pca, kernel='rbf', gamma=None) 
        pca_results = kpca.fit_transform(infra_scaled)
        
        pc_cols = [f'PC{i+1}' for i in range(n_pca)]
        for i, col in enumerate(pc_cols):
            df_temp[col] = pca_results[:, i]
            
        # --- (2) 군집화 ---
        cluster_input_features = base_features + pc_cols
        scaler_final = StandardScaler()
        X_scaled = scaler_final.fit_transform(df_temp[cluster_input_features])
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_temp['cluster'] = kmeans.fit_predict(X_scaled)
        
        # --- (3) 전처리 (원핫 인코딩) ---
        df_encoded = pd.get_dummies(df_temp, columns=['cluster'], prefix='cluster', dtype=int)
        cluster_dummy_cols = [c for c in df_encoded.columns if c.startswith('cluster_')]
        
        final_features = base_features + pc_cols + cluster_dummy_cols
        X = df_encoded[final_features]
        y = df_encoded['월세']
        
        # --- (4) 모델 학습 ---
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
        gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
        hgb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)
        
        voting_model = VotingRegressor(estimators=[('rf', rf), ('gb', gb), ('hgb', hgb)])
        voting_model.fit(X_train, y_train)
        
        # --- (5) 평가 ---
        y_pred = voting_model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)
        
        # 결과 저장
        results_list.append({
            'PCA': n_pca,
            'Cluster': n_clusters,
            'R2 Score': round(r2, 4),
            'MAE': round(mae, 2),
            'RMSE': round(rmse, 2)
        })
        print(f"✅ PCA:{n_pca}, Cluster:{n_clusters} 완료 (R2: {r2:.4f})")

# 3. 요약 결과 출력
summary_df = pd.DataFrame(results_list)
summary_df = summary_df.sort_values(by='R2 Score', ascending=False) # R2 점수 높은 순 정렬

print("\n🏆 [전체 테스트 결과 요약 (R2 높은 순)]")
print(summary_df.to_string(index=False))

# 4. 최적의 조합 안내
best = summary_df.iloc[0]
print(f"\n✨ 최적의 조합: PCA {int(best['PCA'])}개, 군집 {int(best['Cluster'])}개")
print(f"   (R2 Score: {best['R2 Score']}, MAE: {best['MAE']}만원)")

모두 PCA + Clustering 14개의 조합 테스트를 시작합니다. 잠시만 기다려 주세요...

✅ PCA:2, Cluster:2 완료 (R2: 0.6578)
✅ PCA:2, Cluster:3 완료 (R2: 0.6612)
✅ PCA:2, Cluster:4 완료 (R2: 0.6568)
✅ PCA:3, Cluster:2 완료 (R2: 0.6696)
✅ PCA:3, Cluster:3 완료 (R2: 0.6631)
✅ PCA:3, Cluster:4 완료 (R2: 0.6645)
✅ PCA:4, Cluster:2 완료 (R2: 0.6399)
✅ PCA:4, Cluster:3 완료 (R2: 0.6404)
✅ PCA:4, Cluster:4 완료 (R2: 0.6473)
✅ PCA:5, Cluster:2 완료 (R2: 0.6161)
✅ PCA:5, Cluster:3 완료 (R2: 0.6146)
✅ PCA:5, Cluster:4 완료 (R2: 0.6249)
✅ PCA:6, Cluster:2 완료 (R2: 0.6227)
✅ PCA:6, Cluster:3 완료 (R2: 0.6273)
✅ PCA:6, Cluster:4 완료 (R2: 0.6205)
✅ PCA:7, Cluster:2 완료 (R2: 0.6273)
✅ PCA:7, Cluster:3 완료 (R2: 0.6210)
✅ PCA:7, Cluster:4 완료 (R2: 0.6197)
✅ PCA:8, Cluster:2 완료 (R2: 0.6265)
✅ PCA:8, Cluster:3 완료 (R2: 0.6269)
✅ PCA:8, Cluster:4 완료 (R2: 0.6302)

🏆 [전체 테스트 결과 요약 (R2 높은 순)]
 PCA  Cluster  R2 Score  MAE  RMSE
   3        2    0.6696 2.37  3.08
   3        4    0.6645 2.40  3.10
   3        3    0.6631 2.41  3.11
   2        3    0.6612 2.43  3.12
   2  

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import warnings

# 경고 메시지 무시
warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 초기 필터링
df_origin = pd.read_csv("zigbang_outlier.csv")
df_base = df_origin[df_origin['is_outlier'] == False].copy()

# [설정] 군집화와 학습에 사용할 원본 컬럼들
features_cols = [
    '세탁소_거리(m)', '카페_거리(m)', '약국_거리(m)', '대형마트_거리(m)', 
    '편의점_거리(m)', '지하철역_거리(m)', '버스정류장_거리(m)', 
    '보증금', '전용면적', '노후도', '해당층', '전체층', '엘리베이터', '남향', '풀옵션'
]

# 불리언 타입 변환 (0, 1)
for col in ['엘리베이터', '남향', '풀옵션']:
    if col in df_base.columns:
        df_base[col] = df_base[col].astype(int)

# 결과를 저장할 리스트
results_list = []

print("🚀 PCA 없이 '원본 데이터 + 군집화' 조합 테스트를 시작합니다 (Cluster 2~7)...\n")

# 2. 반복문 실행 (Cluster 2~7)
for n_clusters in range(2, 8):
    df_temp = df_base.copy()
    
    # --- (1) 데이터 표준화 (군집화 전 필수) ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_temp[features_cols])
    
    # --- (2) K-Means 군집화 ---
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df_temp['cluster'] = kmeans.fit_predict(X_scaled)
    
    # --- (3) 전처리 (군집 번호 원핫 인코딩) ---
    df_encoded = pd.get_dummies(df_temp, columns=['cluster'], prefix='cluster', dtype=int)
    cluster_dummy_cols = [c for c in df_encoded.columns if c.startswith('cluster_')]
    
    # 최종 모델 학습 변수 설정 (원본 변수 + 군집 더미)
    final_features = features_cols + cluster_dummy_cols
    X = df_encoded[final_features]
    y = df_encoded['월세']
    
    # --- (4) 모델 학습 (앙상블) ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
    gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
    hgb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)
    
    voting_model = VotingRegressor(estimators=[('rf', rf), ('gb', gb), ('hgb', hgb)])
    voting_model.fit(X_train, y_train)
    
    # --- (5) 평가 ---
    y_pred = voting_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    
    # 결과 저장
    results_list.append({
        'Cluster_Count': n_clusters,
        'R2 Score': round(r2, 4),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2)
    })
    print(f"✅ Cluster:{n_clusters} 완료 (R2: {r2:.4f})")

# 3. 요약 결과 출력
summary_df = pd.DataFrame(results_list)
summary_df = summary_df.sort_values(by='R2 Score', ascending=False)

print("\n🏆 [PCA 제외 - 군집수별 테스트 결과 요약]")
print(summary_df.to_string(index=False))

# 4. 최적의 조합 안내
best = summary_df.iloc[0]
print(f"\n✨ 최적의 군집 수: {int(best['Cluster_Count'])}개")
print(f"   (R2 Score: {best['R2 Score']}, MAE: {best['MAE']}만원)")

🚀 PCA 없이 '원본 데이터 + 군집화' 조합 테스트를 시작합니다 (Cluster 2~7)...

✅ Cluster:2 완료 (R2: 0.7254)
✅ Cluster:3 완료 (R2: 0.7224)
✅ Cluster:4 완료 (R2: 0.7206)
✅ Cluster:5 완료 (R2: 0.7242)
✅ Cluster:6 완료 (R2: 0.7219)
✅ Cluster:7 완료 (R2: 0.7237)

🏆 [PCA 제외 - 군집수별 테스트 결과 요약]
 Cluster_Count  R2 Score  MAE  RMSE
             2    0.7254 2.22  2.81
             5    0.7242 2.23  2.81
             7    0.7237 2.23  2.81
             3    0.7224 2.24  2.82
             6    0.7219 2.23  2.82
             4    0.7206 2.24  2.83

✨ 최적의 군집 수: 2개
   (R2 Score: 0.7254, MAE: 2.22만원)
